# 01 — Retrieval fundamentals: dense, BM25, and Reciprocal Rank Fusion

Companion notebook to the [Block 1 course](../README.md). Everything here runs on a **toy corpus of
12 financial sentences**, so every number can be checked by eye.

What you will see:

1. **Dense retrieval**: embed text, rank by cosine similarity.
2. **BM25 by hand**: compute one score term by term, then check it against `ragbench.lexical.BM25`
   and the reference library `rank-bm25`.
3. **Where each method fails**: a paraphrase that BM25 misses, an acronym that *both* miss, and
   near-duplicate identifiers where dense is right but with a dangerously thin margin.
4. **Reciprocal Rank Fusion**: merge the two rankings with `ragbench.fusion.rrf_fuse`.

> Scope warning: a 12-sentence corpus shows *mechanisms*, not *performance*. Performance is
> measured in Block 2 on thousands of FinQA chunks, with confidence intervals.

In [1]:
import math
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from ragbench.fusion import rrf_fuse
from ragbench.lexical import BM25, tokenize

np.set_printoptions(precision=3, suppress=True)

## The toy corpus

Three groups are built on purpose:

- **0–2**: three accounting-standard adoptions (ASC 842, ASC 606, ASU 2016-13). The last one is
  the "CECL" standard, but the acronym *CECL* never appears in the text.
- **3–5**: three credit facilities whose sentences are identical except for an internal code and
  the amounts. `KX-4471` and `KX-4417` differ only by swapped digits.
- **6–11**: generic financial facts.

In [2]:
CORPUS = [
    "We adopted ASC 842 on January 1, 2019, recognizing right-of-use assets for operating leases.",
    "We adopted ASC 606 on January 1, 2018, changing the timing of revenue recognition on service contracts.",
    "We adopted ASU 2016-13 on January 1, 2020, introducing the current expected credit loss model.",
    "Under credit facility KX-4471 the company may borrow up to $500 million at SOFR plus 1.25%.",
    "Under credit facility KX-4417 the company may borrow up to $350 million at SOFR plus 1.50%.",
    "Under credit facility RV-2093 the company may borrow up to $200 million at SOFR plus 1.75%.",
    "Net revenue grew 3% in the second quarter, driven by strong demand in the Americas.",
    "The company repurchased 2.1 million shares of common stock during the year.",
    "Goodwill impairment of $45 million was recorded for the retail reporting unit.",
    "The effective tax rate was 21%, compared with 24% in the prior year.",
    "Free cash flow improved thanks to lower capital expenditures.",
    "Our Tier 1 capital ratio remained well above regulatory minimums under Basel III.",
]
for i, doc in enumerate(CORPUS):
    print(f"[{i:2}] {doc}")

[ 0] We adopted ASC 842 on January 1, 2019, recognizing right-of-use assets for operating leases.
[ 1] We adopted ASC 606 on January 1, 2018, changing the timing of revenue recognition on service contracts.
[ 2] We adopted ASU 2016-13 on January 1, 2020, introducing the current expected credit loss model.
[ 3] Under credit facility KX-4471 the company may borrow up to $500 million at SOFR plus 1.25%.
[ 4] Under credit facility KX-4417 the company may borrow up to $350 million at SOFR plus 1.50%.
[ 5] Under credit facility RV-2093 the company may borrow up to $200 million at SOFR plus 1.75%.
[ 6] Net revenue grew 3% in the second quarter, driven by strong demand in the Americas.
[ 7] The company repurchased 2.1 million shares of common stock during the year.
[ 8] Goodwill impairment of $45 million was recorded for the retail reporting unit.
[ 9] The effective tax rate was 21%, compared with 24% in the prior year.
[10] Free cash flow improved thanks to lower capital expenditures.
[11] Ou

## 1. Dense retrieval

A **bi-encoder** maps each text to a vector (here 384 dimensions) independently. Queries and
documents never "see" each other; relevance is approximated by **cosine similarity** between
vectors. Because vectors are L2-normalized, cosine similarity is just a dot product:

$$\cos(q, d) = \frac{q \cdot d}{\lVert q \rVert \, \lVert d \rVert} = q \cdot d \quad\text{when } \lVert q \rVert = \lVert d \rVert = 1$$

BGE models are trained with an **instruction prefix on the query side only**. Forgetting it is a
classic silent bug: retrieval still "works", just worse.

In [3]:
import transformers

transformers.logging.disable_progress_bar()
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

doc_emb = model.encode(CORPUS, normalize_embeddings=True)
print("embedding matrix:", doc_emb.shape, "| norms:", np.linalg.norm(doc_emb, axis=1)[:3], "...")


def dense_rank(query: str) -> tuple[list[int], np.ndarray]:
    q = model.encode(QUERY_PREFIX + query, normalize_embeddings=True)
    scores = doc_emb @ q
    return [int(i) for i in np.argsort(-scores)], scores

embedding matrix: (12, 384) | norms: [1. 1. 1.] ...


## 2. BM25, by hand then by code

For a query $q$ and a document $d$:

$$\text{BM25}(q,d)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{tf(t,d)\,(k_1+1)}{tf(t,d)+k_1\left(1-b+b\,\frac{|d|}{\text{avgdl}}\right)}
\qquad
\text{IDF}(t)=\ln\frac{N-n_t+0.5}{n_t+0.5}$$

We compute the score of document 4 for the query `"KX-4417 borrowing limit"` term by term.

In [4]:
tokens = [tokenize(d) for d in CORPUS]
query = tokenize("KX-4417 borrowing limit")
print("query tokens:", query)

k1, b, N = 1.5, 0.75, len(tokens)
avgdl = sum(map(len, tokens)) / N
doc = tokens[4]
print(f"N={N}  avgdl={avgdl:.2f}  |d4|={len(doc)}")

total = 0.0
for t in query:
    n_t = sum(t in d for d in tokens)
    tf = doc.count(t)
    idf = math.log((N - n_t + 0.5) / (n_t + 0.5)) if n_t else 0.0
    part = idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len(doc) / avgdl)) if tf else 0.0
    total += part
    print(f"  term={t!r:10} n_t={n_t}  tf={tf}  idf={idf:6.3f}  contribution={part:6.3f}")
print(f"hand-computed BM25(q, d4) = {total:.4f}")

ours = BM25(tokens).score(query, 4)
ref = BM25Okapi(tokens).get_scores(query)[4]
print(f"ragbench.BM25            = {ours:.4f}")
print(f"rank_bm25.BM25Okapi      = {ref:.4f}")

query tokens: ['kx', '4417', 'borrowing', 'limit']
N=12  avgdl=14.83  |d4|=18
  term='kx'       n_t=2  tf=1  idf= 1.435  contribution= 1.309
  term='4417'     n_t=1  tf=1  idf= 2.037  contribution= 1.858
  term='borrowing' n_t=0  tf=0  idf= 0.000  contribution= 0.000
  term='limit'    n_t=0  tf=0  idf= 0.000  contribution= 0.000
hand-computed BM25(q, d4) = 3.1677
ragbench.BM25            = 3.1677
rank_bm25.BM25Okapi      = 3.1677


Read the contributions: the tokenizer splits `KX-4417` into `kx` and `4417`. The token `kx`
appears in two documents (3 and 4), so it is only mildly informative; `4417` appears **once**
in the corpus and carries most of the score. Words absent from the document (`borrowing`,
`limit`) contribute nothing: **BM25 cannot match what is not literally written**.

(If some terms had appeared in more than half the documents, the hand computation would differ
from the library: `rank-bm25` replaces negative IDFs by a floor. See the course, section 2.3.)

## 3. Where each method fails

We compare both rankers on five queries. For each we print the top-3 documents and their scores,
plus the **margin** between rank 1 and rank 2: the smaller it is, the more fragile the decision
once thousands of competing chunks enter the index.

In [5]:
bm25 = BM25(tokens)
CASES = {
    "share buyback": 7,              # paraphrase: corpus says "repurchased ... shares"
    "CECL adoption": 2,              # acronym never written in the corpus
    "KX-4417 borrowing limit": 4,    # near-duplicate identifier (KX-4471 exists)
    "RV-2093 facility": 5,
    "revenue recognition standard": 1,
}


def show(name, ranking, scores, gold):
    top = ranking[:3]
    s = [round(float(scores[i]), 3) for i in top]
    margin = float(scores[top[0]] - scores[top[1]])
    hit = "HIT " if top[0] == gold else "MISS"
    print(f"   {name:5} {hit} top3={top} scores={s} margin={margin:.3f}")


for q, gold in CASES.items():
    print(f"{q!r} (gold = doc {gold})")
    d_rank, d_scores = dense_rank(q)
    show("dense", d_rank, d_scores, gold)
    b_scores = np.array(bm25.get_scores(tokenize(q)))
    b_rank = [int(i) for i in np.argsort(-b_scores, kind="stable")]
    show("bm25", b_rank, b_scores, gold)

'share buyback' (gold = doc 7)
   dense HIT  top3=[7, 4, 3] scores=[0.66, 0.592, 0.584] margin=0.067
   bm25  MISS top3=[0, 1, 2] scores=[0.0, 0.0, 0.0] margin=0.000
'CECL adoption' (gold = doc 2)
   dense MISS top3=[0, 1, 2] scores=[0.589, 0.56, 0.538] margin=0.029
   bm25  MISS top3=[0, 1, 2] scores=[0.0, 0.0, 0.0] margin=0.000
'KX-4417 borrowing limit' (gold = doc 4)
   dense HIT  top3=[4, 3, 5] scores=[0.741, 0.7, 0.646] margin=0.041
   bm25  HIT  top3=[4, 3, 0] scores=[3.168, 1.309, 0.0] margin=1.858
'RV-2093 facility' (gold = doc 5)
   dense HIT  top3=[5, 3, 4] scores=[0.652, 0.509, 0.471] margin=0.143
   bm25  HIT  top3=[5, 3, 4] scores=[4.628, 0.911, 0.911] margin=3.717
'revenue recognition standard' (gold = doc 1)
   dense HIT  top3=[1, 0, 2] scores=[0.673, 0.611, 0.545] margin=0.062
   bm25  HIT  top3=[1, 6, 0] scores=[3.258, 1.428, 0.0] margin=1.830


What to take away (check it against the output above):

- **`share buyback`**: BM25 scores **0 for every document**, since none contains "share" + "buyback"
  in that form (the gold says *repurchased ... shares*, and the tokenizer does not stem
  "share"/"shares"). Its "ranking" is just corpus order. Dense finds the gold: it matches meaning.
- **`CECL adoption`**: **both fail**. The text says "current expected credit loss model" but never
  "CECL". Dense does not know the acronym well enough, and BM25 has nothing to match. This is
  exactly the gap **Contextual Retrieval** targets: an LLM-written context such as *"This chunk
  describes the adoption of ASU 2016-13 (CECL)..."* adds the missing words *before* indexing.
- **`KX-4417 borrowing limit`**: dense is right, but compare the margins: dense separates
  KX-4417 from KX-4471 by a few hundredths of cosine, while BM25 separates them by a factor of
  ~2.4 (3.17 vs 1.31), because the rare token `4417` is decisive. On 12 sentences the thin margin is enough; on a
  pooled index of thousands of near-identical filing chunks, it is the kind of margin noise can
  overturn. **That is the argument for hybrid search**, and Block 2 tests it with numbers.

## 4. Reciprocal Rank Fusion

RRF merges rankings using **ranks only**: $\text{RRF}(d)=\sum_L 1/(k+\text{rank}_L(d))$, with $k=60$.
BM25 scores (0 to ~5 here) and cosine similarities (~0.4 to 0.75) live on incompatible scales;
ranks do not.

One subtlety: when BM25 scores **all** documents 0, its "ranking" is arbitrary. Feeding it to RRF
injects noise. The sensible thing is to pass only documents with a **positive** BM25 score.

In [6]:
def hybrid(q: str, k: int = 60, depth: int = 5):
    d_rank, _ = dense_rank(q)
    b_rank = [i for i, s in bm25.top_k(tokenize(q), depth) if s > 0]
    return d_rank[:depth], b_rank, rrf_fuse([d_rank[:depth], b_rank], k=k)


for q, gold in CASES.items():
    d, b_, fused = hybrid(q)
    top = [doc for doc, _ in fused[:3]]
    print(f"{q!r:32} dense={d[:3]} bm25={b_[:3]} -> rrf={top}  {'HIT' if top[0] == gold else 'MISS'}")

'share buyback'                  dense=[7, 4, 3] bm25=[] -> rrf=[7, 4, 3]  HIT
'CECL adoption'                  dense=[0, 1, 2] bm25=[] -> rrf=[0, 1, 2]  MISS
'KX-4417 borrowing limit'        dense=[4, 3, 5] bm25=[4, 3] -> rrf=[4, 3, 5]  HIT
'RV-2093 facility'               dense=[5, 3, 4] bm25=[5, 3, 4] -> rrf=[5, 3, 4]  HIT
'revenue recognition standard'   dense=[1, 0, 2] bm25=[1, 6] -> rrf=[1, 0, 6]  HIT


RRF keeps the dense answer for the paraphrase (BM25 contributes nothing there) and agrees with
both rankers when they agree. It cannot rescue `CECL adoption`: **fusion only recombines what
the retrievers found**. When the right chunk is in neither list, you need a better
representation (contextual chunks) or a better query (rewriting, agentic search).

### The role of the constant k

With a small $k$, being ranked first is worth much more than being ranked second. With a large $k$,
the gap shrinks and *consensus* between lists matters more:

In [7]:
for k in (0, 10, 60, 1000):
    r1, r2, r10 = 1 / (k + 1), 1 / (k + 2), 1 / (k + 10)
    print(f"k={k:5}  rank1={r1:.4f}  rank2={r2:.4f}  rank10={r10:.4f}  rank1/rank10 = {r1 / r10:5.2f}x")

k=    0  rank1=1.0000  rank2=0.5000  rank10=0.1000  rank1/rank10 = 10.00x
k=   10  rank1=0.0909  rank2=0.0833  rank10=0.0500  rank1/rank10 =  1.82x
k=   60  rank1=0.0164  rank2=0.0161  rank10=0.0143  rank1/rank10 =  1.15x
k= 1000  rank1=0.0010  rank2=0.0010  rank10=0.0010  rank1/rank10 =  1.01x


## Summary

| Method | Matches | Fails on | Cost |
|---|---|---|---|
| Dense (bi-encoder) | meaning, paraphrases | rare identifiers (thin margins), unknown acronyms | one embedding per query |
| BM25 | exact rare tokens, codes, numbers | paraphrases, synonyms, inflections | nearly free |
| RRF (hybrid) | the union of both | what *neither* retriever surfaced | negligible |

Next: the course explains how **Contextual Retrieval** (fix the representation) and **reranking**
(fix the ordering) push further, and Block 2 measures all of it on FinQA.